In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')

customers.head()
customers.shape
customers.info()
customers.describe(include="all")
customers.isnull().sum()
customers.duplicated().sum()
customers['customer_id'].duplicated().sum()
customers['customer_unique_id'].duplicated().sum()
customers['customer_state'].value_counts()
customers['customer_city'].nunique()
customers['customer_city'].sample(20)

customers.to_csv('../data/processed/customers_cleaned.csv', index=False)

## Customers Dataset Cleaning Summary

### Dataset Overview
- Rows: 99,441
- Columns: 5

### Data Quality Checks
- No missing values found.
- No duplicate rows found.
- `customer_id` is unique for every record.
- `customer_unique_id` contains repeated values, which is expected because a single customer can place multiple orders.
- Customer city and state values appear consistent.
- No datatype conversion required.

### Cleaning Performed
No cleaning was required for this dataset. It is ready for analysis and loading into the analytical database.

In [ ]:
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')

orders.head()
orders.shape
orders.info()
orders.describe(include="all")
orders.isnull().sum()
orders.duplicated().sum()
orders['order_id'].duplicated().sum()
orders['order_status'].value_counts()

date_columns = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

orders.groupby('order_status').agg({
    'order_delivered_customer_date': lambda x: x.isna().sum(),
    'order_delivered_carrier_date': lambda x: x.isna().sum()
})

(
    orders['order_delivered_customer_date'] 
    < orders['order_purchase_timestamp']
).sum()
(
    orders['order_approved_at']
    < orders['order_purchase_timestamp']
).sum()
(
    orders['order_estimated_delivery_date']
    < orders['order_purchase_timestamp']
).sum()

orders.info()
orders.isnull().sum()
orders['order_status'].value_counts()

orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].isnull())
]

# Total orders
orders['order_id'].nunique()

# Total customers
orders['customer_id'].nunique()

# Order status distribution
orders['order_status'].value_counts(normalize=True) * 100

#Data Range
orders['order_purchase_timestamp'].min(), orders['order_purchase_timestamp'].max()

orders.to_csv('../data/processed/orders_cleaned.csv', index=False)

## Orders Dataset Cleaning Summary

### Dataset Overview
- Rows: 99,441
- Columns: 8
- Central transactional table of the Olist dataset.

### Data Quality Checks
- No duplicate rows.
- `order_id` is unique.
- Converted all timestamp columns to datetime.
- Missing values in approval and delivery dates are expected for orders that were canceled, unavailable, or still in progress.
- Identified two delivered orders with missing delivery timestamps for further review.

### Cleaning Performed
- Converted timestamp columns to datetime.
- Preserved business-related missing values.
- No rows removed.
- Dataset is ready for analysis.

### Additional Observations
- Identified 8 orders marked as **delivered** but missing the actual customer delivery timestamp.
- One of these orders is also missing the carrier delivery timestamp.
- These records represent data inconsistencies in the source dataset.
- Since they account for less than 0.01% of all records, they were retained and documented rather than removed.

In [ ]:
products = pd.read_csv('../data/raw/olist_products_dataset.csv')

products.head()
products.shape
products.info()
products.isnull().sum()
products.duplicated().sum()

products[
    products['product_category_name'].isnull()
]
products[
    products['product_weight_g'].isnull()
]

integer_columns = [
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty'
]

products[integer_columns] = products[integer_columns].astype('Int64')

products.info()
products.isnull().sum()
products[
    products['product_category_name'].isnull()
].head()

products.to_csv('../data/processed/products_cleaned.csv', index=False)

## Products Dataset Cleaning Summary

### Dataset Overview
- Rows: 32,951
- Columns: 9
- Primary Key: `product_id`

### Data Quality Checks
- No duplicate rows found.
- `product_id` is unique.
- 610 products are missing category and descriptive information.
- 2 products are missing physical dimensions (weight and size).

### Cleaning Decisions
- Converted `product_name_lenght`, `product_description_lenght`, and `product_photos_qty` to nullable integer (`Int64`) data type.
- Preserved all missing values to maintain source data integrity.
- No records were removed because missing values represent a small portion of the dataset and deleting them could affect downstream analyses.

### Outcome
The dataset is ready for integration with transactional tables. Missing values will be handled during analysis or visualization when necessary.

In [ ]:
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')

order_items.head()
order_items.shape
order_items.info()
order_items.isnull().sum()
order_items.duplicated().sum()
order_items.duplicated(subset=['order_id', 'order_item_id']).sum()

order_items['shipping_limit_date'] = pd.to_datetime(
    order_items['shipping_limit_date']
)

order_items['price'].describe()
order_items['price'].sort_values(ascending=False).head(10)
order_items['freight_value'].describe()
order_items['freight_value'].sort_values(ascending=False).head(10)
(order_items['price'] <= 0).sum()
(order_items['freight_value'] < 0).sum()
order_items.groupby('order_id').size().sort_values(
    ascending = False
).head(10)

order_items.info()
order_items.isnull().sum()
order_items.describe()
(order_items['price'] <= 0).sum()
(order_items['freight_value'] < 0).sum()

order_items.to_csv('../data/processed/order_items_cleaned.csv', index=False)

## Order Items Dataset Cleaning Summary

### Dataset Overview
- Rows: 112,650
- Columns: 7
- Composite Primary Key: (`order_id`, `order_item_id`)

### Data Quality Checks
- No missing values found.
- No duplicate rows found.
- Composite primary key is unique.
- Converted `shipping_limit_date` to datetime format.
- Validated product prices and freight values for non-negative values.

### Cleaning Performed
- Converted shipping limit date to datetime.
- No records removed.
- Dataset is clean and ready for integration.

### Business Importance
This table contains the transactional sales information for each product sold and will serve as the foundation for the analytical fact table used in SQL analysis and Power BI dashboards.

In [ ]:
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')

payments.head()
payments.shape
payments.info()
payments.isnull().sum()
payments.duplicated().sum()
payments.duplicated(subset = ['order_id', 'payment_sequential']).sum()
payments['payment_type'].value_counts()
payments['payment_installments'].describe()
payments['payment_installments'].value_counts().sort_index()
payments['payment_value'].describe()
payments.nlargest(10, 'payment_value')
(payments['payment_value'] < 0).sum()
(payments['payment_value'] == 0).sum()
payments[payments['payment_value'] == 0]
(payments['payment_installments'] <= 0).sum()

payments[payments['payment_installments'] <= 0]
payments[payments['order_id'] == '744bade1fcf9ff3f31d860ace076d422']
payments[payments['order_id'] == '1a57108394169c0b47d8f876acc9ba2d']
payments[
    payments['order_id'].isin([
        '744bade1fcf9ff3f31d860ace076d422', 
        '1a57108394169c0b47d8f876acc9ba2d'
    ])
].sort_values(['order_id', 'payment_sequential'])

payments.groupby('order_id').size().sort_values(
   ascending=False
).head(10)

payments.to_csv('../data/processed/payments_cleaned.csv', index=False)

## Payments Dataset Cleaning Summary

### Additional Observations

During data validation, two anomalous payment records were identified.

#### Data Quality Issue

- Both records have `payment_installments = 0` despite using the `credit_card` payment type.
- Both records also have `payment_sequential = 2`, while no corresponding `payment_sequential = 1` record exists for the same order.

These characteristics suggest inconsistencies in the original source data rather than issues introduced during preprocessing.

#### Impact Analysis

- Total payment records: **103,886**
- Anomalous records: **2**
- Percentage affected: **0.0019%**

The anomalies represent an extremely small proportion of the dataset and are unlikely to influence aggregate analyses.

#### Cleaning Decision

The records were retained without modification because:

- There is no authoritative source indicating the correct values.
- Altering payment sequence or installment counts would require assumptions.
- Preserving the original data ensures reproducibility and maintains source data integrity.

Future analyses involving installment behavior can exclude these two records if necessary.